# 01 Data Loading and Preprocessing

This notebook contains the first stage of the project workflow.

Main purpose:
- Import required Python libraries
- Load Queensland crash-related datasets
- Inspect dataset columns and basic statistics
- Check missing values and duplicate records
- Merge crash location data with road roughness data using KDTree
- Prepare the combined dataset for later analysis and modelling

> Note: This version keeps the original code unchanged. Only explanatory Markdown sections have been added for GitHub readability.

## 1. Import Libraries

This cell imports the Python libraries used across the project, including data processing, machine learning, visualisation, spatial matching and clustering tools.

In [49]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, ShuffleSplit, learning_curve
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, silhouette_score
from imblearn.over_sampling import SMOTE
from keras.models import Sequential
from keras.layers import Dense, Dropout
from keras.optimizers import Adam
from geopy.distance import great_circle
from shapely.geometry import MultiPoint
from scipy.spatial import cKDTree
from sklearn.cluster import DBSCAN



## 2. Load Raw CSV Files

This cell loads the crash datasets and road condition datasets into pandas DataFrames. It also prints column names to confirm that the files have been loaded successfully.

In [50]:

#load all csv file
df_CL = pd.read_csv("crash_locations.csv")
df_DI = pd.read_csv("driver_involvement.csv")
df_FIRC = pd.read_csv("factor_in_road_crash.csv")
df_HU = pd.read_csv("helmet_use.csv")
df_RC = pd.read_csv("road_casualties.csv")
df_VI = pd.read_csv("vehicle_involvement.csv")
df_hf = pd.read_csv("high_rainfall.csv")
df_r100 = pd.read_csv("roughness_100m.csv")
file_list=[df_CL,df_DI,df_FIRC,df_HU,df_RC,df_VI]
#check the file successful load
for x in file_list:
    column_names = x.columns.tolist()
    print(column_names)
print(df_CL.head())
column_names=df_hf.columns.tolist()
print(column_names)

C:\Users\29084\AppData\Local\Temp\ipykernel_6368\2719910160.py:2: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  df_CL = pd.read_csv("crash_locations.csv")


['Crash_Ref_Number', 'Crash_Severity', 'Crash_Year', 'Crash_Month', 'Crash_Day_Of_Week', 'Crash_Hour', 'Crash_Nature', 'Crash_Type', 'Crash_Longitude', 'Crash_Latitude', 'Crash_Street', 'Crash_Street_Intersecting', 'State_Road_Name', 'Loc_Suburb', 'Loc_Local_Government_Area', 'Loc_Post_Code', 'Loc_Police_Division', 'Loc_Police_District', 'Loc_Police_Region', 'Loc_Queensland_Transport_Region', 'Loc_Main_Roads_Region', 'Loc_ABS_Statistical_Area_2', 'Loc_ABS_Statistical_Area_3', 'Loc_ABS_Statistical_Area_4', 'Loc_ABS_Remoteness', 'Loc_State_Electorate', 'Loc_Federal_Electorate', 'Crash_Controlling_Authority', 'Crash_Roadway_Feature', 'Crash_Traffic_Control', 'Crash_Speed_Limit', 'Crash_Road_Surface_Condition', 'Crash_Atmospheric_Condition', 'Crash_Lighting_Condition', 'Crash_Road_Horiz_Align', 'Crash_Road_Vert_Align', 'Crash_DCA_Code', 'Crash_DCA_Description', 'Crash_DCA_Group_Description', 'DCA_Key_Approach_Dir', 'Count_Casualty_Fatality', 'Count_Casualty_Hospitalised', 'Count_Casualty_M

## 3. Initial Dataset Inspection

This cell provides a basic statistical summary of the crash location dataset and checks its number of rows and columns.

In [13]:
print(df_CL.describe(include='all'))
rows, columns = df_CL.shape
print(f" {rows}row and {columns} column")

        Crash_Ref_Number     Crash_Severity     Crash_Year Crash_Month  \
count      380316.000000             380316  380316.000000      380316   
unique               NaN                  5            NaN          12   
top                  NaN  Medical treatment            NaN      August   
freq                 NaN             118909            NaN       33865   
mean       190158.500000                NaN    2009.946839         NaN   
std        109787.916826                NaN       6.230248         NaN   
min             1.000000                NaN    2001.000000         NaN   
25%         95079.750000                NaN    2005.000000         NaN   
50%        190158.500000                NaN    2009.000000         NaN   
75%        285237.250000                NaN    2015.000000         NaN   
max        380316.000000                NaN    2023.000000         NaN   

       Crash_Day_Of_Week     Crash_Hour Crash_Nature     Crash_Type  \
count             380316  380316.000000 

## 4. Learning Curve Helper Function

This cell defines a helper function for plotting learning curves. It is used later to understand model behaviour across different training set sizes.

In [14]:
#learning curve to check the  DF_HU and DF_RC dataframe(feature select the frist one is Crash_Year),
#maybe delete Crash_year column to recheck(5.5)

def plot_learning_curve(estimator, title, X, y, ylim=None, cv=None,
                        n_jobs=None, train_sizes=np.linspace(.1, 1.0, 5)):
    plt.figure()
    plt.title(title)
    if ylim is not None:
        plt.ylim(*ylim)
    plt.xlabel("Training examples")
    plt.ylabel("Score")
    train_sizes, train_scores, test_scores = learning_curve(
        estimator, X, y, cv=cv, n_jobs=n_jobs, train_sizes=train_sizes)
    train_scores_mean = np.mean(train_scores, axis=1)
    train_scores_std = np.std(train_scores, axis=1)
    test_scores_mean = np.mean(test_scores, axis=1)
    test_scores_std = np.std(test_scores, axis=1)
    plt.grid()

    plt.fill_between(train_sizes, train_scores_mean - train_scores_std,
                     train_scores_mean + train_scores_std, color="r", alpha=0.1)
    plt.fill_between(train_sizes, test_scores_mean - test_scores_std,
                     test_scores_mean + test_scores_std, color="g", alpha=0.1)
    plt.plot(train_sizes, train_scores_mean, 'o-', color="r",
             label="Training score")
    plt.plot(train_sizes, test_scores_mean, 'o-', color="g",
             label="Cross-validation score")
    plt.legend(loc="best")
    return plt

# need check later and understand that part

## 5. Data Cleaning: Missing Values and Duplicates

This cell removes missing values and checks duplicate records before the spatial data merge.

In [15]:
# before the combination to the data clean, replicate and null value
df_CL.dropna(inplace=True)
duplicates = df_CL.duplicated()
print("Number of duplicate rows:", duplicates.sum())
# check the duplicate number
if duplicates.sum() > 0:
    print(df_CL[duplicates])
df_r100.dropna(inplace=True)# choose 100m road section, 1000m may not accurate(3.11)
duplicates = df_r100.duplicated()
print("Number of duplicate rows:", duplicates.sum())
if duplicates.sum() > 0:
    print(df_r100[duplicates])


Number of duplicate rows: 0
Number of duplicate rows: 0


## 6. Spatial Matching with KDTree

This cell matches crash locations with the nearest road roughness records based on latitude and longitude. KDTree is used to make nearest-neighbour search more efficient.

In [16]:
# based on the latitude and longitude combination
# Convert the relevant columns to a numpy array for faster computation
coords1 = df_CL[['Crash_Latitude', 'Crash_Longitude']].to_numpy()
coords2 = df_r100[['Latitude', 'Longitude']].to_numpy()

# Create a KDTree 
tree = cKDTree(coords2)
#for each point in dataset1 to find the closest point in dataset2
distances, indices = tree.query(coords1, k=1)
# 0.01 maximum distance 
max_distance = 0.01
valid_matches = distances <= max_distance

# Use closest points to merge the datasets
matched_dataset1 = df_CL[valid_matches]
matched_dataset2 = df_r100.iloc[indices[valid_matches]]
# Resetting index to allow for a clean concatenation
matched_dataset1.reset_index(drop=True, inplace=True)
matched_dataset2.reset_index(drop=True, inplace=True)
# Combining the datasets. 
combined_dataset = pd.concat([matched_dataset1, matched_dataset2], axis=1)
print(combined_dataset.columns)


Index(['Crash_Ref_Number', 'Crash_Severity', 'Crash_Year', 'Crash_Month',
       'Crash_Day_Of_Week', 'Crash_Hour', 'Crash_Nature', 'Crash_Type',
       'Crash_Longitude', 'Crash_Latitude', 'Crash_Street',
       'Crash_Street_Intersecting', 'State_Road_Name', 'Loc_Suburb',
       'Loc_Local_Government_Area', 'Loc_Post_Code', 'Loc_Police_Division',
       'Loc_Police_District', 'Loc_Police_Region',
       'Loc_Queensland_Transport_Region', 'Loc_Main_Roads_Region',
       'Loc_ABS_Statistical_Area_2', 'Loc_ABS_Statistical_Area_3',
       'Loc_ABS_Statistical_Area_4', 'Loc_ABS_Remoteness',
       'Loc_State_Electorate', 'Loc_Federal_Electorate',
       'Crash_Controlling_Authority', 'Crash_Roadway_Feature',
       'Crash_Traffic_Control', 'Crash_Speed_Limit',
       'Crash_Road_Surface_Condition', 'Crash_Atmospheric_Condition',
       'Crash_Lighting_Condition', 'Crash_Road_Horiz_Align',
       'Crash_Road_Vert_Align', 'Crash_DCA_Code', 'Crash_DCA_Description',
       'Crash_DCA_Group_De

## 7. Load Combined Dataset and Prepare Grouped Output

This cell loads the combined dataset and creates a grouped summary by crash severity, roughness class and coordinates for later analysis.

In [17]:
df = pd.read_csv('combined_dataset.csv')
#print(df.columns)
#use combined dataset to draw the graph
#use tableau draw(4.18)
result = df.groupby(['Crash_Severity', 'RoughnessClass', 'Crash_Latitude', 'Crash_Longitude']).size().reset_index(name='count')
#clean null value
df.dropna(inplace=True)
df_cd= df
print(result)



              Crash_Severity RoughnessClass  Crash_Latitude  Crash_Longitude  \
0                      Fatal      0 No data      -28.675155       151.920917   
1                      Fatal      0 No data      -28.395276       150.314120   
2                      Fatal      0 No data      -28.122449       152.067184   
3                      Fatal      0 No data      -28.122367       152.067224   
4                      Fatal      0 No data      -28.082260       153.359771   
...                      ...            ...             ...              ...   
280458  Property damage only   6 Unsurfaced      -14.270510       143.301782   
280459  Property damage only   6 Unsurfaced      -13.614088       143.054465   
280460  Property damage only   6 Unsurfaced      -13.085166       142.742450   
280461  Property damage only   6 Unsurfaced      -12.981323       142.443329   
280462  Property damage only   6 Unsurfaced      -12.691571       142.334394   

        count  
0           1  
1      